# Semana 2: Introducción a Spark con Scala

## Contexto del Ejercicio: Mi Porfolio como Data Engineer

Este notebook forma parte de la actividad **"Mi porfolio como Data Engineer"** descrita en el syllabus (páginas 18-19). 

El objetivo es continuar la construcción de tu **base de conocimiento** (porfolio). Puedes utilizar este notebook como:
*   **Guía**: Para entender la arquitectura de Spark y su API.
*   **Base**: Para experimentar con diferentes transformaciones y acciones.
*   **Complemento**: A tu repositorio de código en GitHub.

---

En esta segunda semana, nos adentraremos en el desarrollo de aplicaciones distribuidas utilizando Apache Spark. Veremos los conceptos fundamentales, trabajaramos con RDDs (Spark Core) y DataFrames (Spark SQL).

## 1. Desarrollo de aplicaciones con Apache Spark

Apache Spark es un motor de análisis unificado para el procesamiento de datos a gran escala.

### Conceptos Clave
*   **Driver**: El proceso principal que ejecuta tu aplicación (el `main`), crea el `SparkContext`/`SparkSession` y coordina las tareas.
*   **Executor**: Procesos que se ejecutan en los nodos del clúster, responsables de ejecutar las tareas y almacenar datos en memoria o disco.
*   **Cluster Manager**: Gestor de recursos (e.g., Standalone, YARN, Kubernetes) que asigna recursos a la aplicación.

### Ventajas
*   **Velocidad**: Ejecución en memoria, mucho más rápido que MapReduce tradicional.
*   **Facilidad de uso**: APIs de alto nivel en Scala, Java, Python y R.
*   **Unificado**: Soporta SQL, Streaming, ML y Graph en un solo motor.

In [1]:
import org.apache.spark.sql.SparkSession

// Inicialización de SparkSession (El punto de entrada a Spark)
val spark = SparkSession.builder()
  .appName("Semana2_Porfolio")
  .master("local[*]") // Ejecutar localmente usando todos los cores disponibles
  //.master("spark://spark-master:7077") // Si activas este modo obtendrás algunos errores por la integración de Ammonite y Spark
  // Memoria del Driver (donde se recolectan los resultados de .collect())
  .config("spark.driver.memory", "2g") 
  // Memoria de cada Executor
  .config("spark.executor.memory", "2g")
  // Memoria adicional por encima del heap (útil para evitar errores de overhead)
  .config("spark.executor.memoryOverhead", "512m")
  //.config("deploy-mode","client") 
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") // Reducir el ruido en los logs

println(s"Spark Version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/15 23:12:02 INFO SparkContext: Running Spark version 4.1.1
26/02/15 23:12:02 INFO SparkContext: OS info Linux, 6.12.54-linuxkit, aarch64
26/02/15 23:12:02 INFO SparkContext: Java version 17.0.18+8-Ubuntu-122.04.1
26/02/15 23:12:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/15 23:12:02 INFO ResourceUtils: ==============================================================
26/02/15 23:12:02 INFO ResourceUtils: No custom resources configured for spark.driver.
26/02/15 23:12:02 INFO ResourceUtils: ==============================================================
26/02/15 23:12:02 INFO SparkContext: Submitted application: Semana2_Porfolio
26/02/15 23:12:02 INFO SecurityManager: Changing view acls to: jovyan
26/02/15 23:12:02 INFO SecurityManager: Changing modify acls to: jovyan
26/02/15 23:12:02 INFO SecurityManager: Changing 

Spark Version: 4.1.1


import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@8444b8b

In [3]:
// Obtenemos el estado de los ejecutores
// Retorna un Map: "host:puerto" -> (Memoria Total, Memoria Libre)
val status = spark.sparkContext.getExecutorMemoryStatus

val hosts = status.keys.toSeq
val totalMem = status.values.map(_._1 / (1024 * 1024)).toSeq // Convertir a MB

println(s"Nodos activos detectados: ${hosts.size}")

Nodos activos detectados: 1


status: collection.Map[String, (Long, Long)] = Map(
  "ec18c6b085c1:39801" -> (1044381696L, 1044381696L)
)
hosts: Seq[String] = List("ec18c6b085c1:39801")
totalMem: Seq[Long] = List(996L)

In [4]:
// En el Driver (Jupyter)
println(System.getProperty("java.version"))

// En los Workers
spark.sparkContext.parallelize(Seq(1)).map(_ => System.getProperty("java.version")).collect().foreach(println)

17.0.18
17.0.18


## 2. Introducción al módulo Spark Core (RDDs)

RDD (Resilient Distributed Dataset) es la abstracción fundamental de Spark. Representa una colección inmutable de objetos distribuida y tolerante a fallos.

### Transformaciones vs Acciones
*   **Transformaciones (Lazy)**: Crean un nuevo RDD a partir de uno existente (ej. `map`, `filter`). No se ejecutan inmediatamente.
*   **Acciones**: Disparan la computación y devuelven un resultado al Driver o guardan datos (ej. `count`, `collect`, `saveAsTextFile`).

In [5]:
// Ejemplo Spark Core: Procesamiento de texto básico con RDDs
val datos = Seq("Spark es rapido", "Spark es genial", "Scala y Spark", "Big Data es el futuro")

// 1. Crear RDD paralelizando una colección existente
val rdd = spark.sparkContext.parallelize(datos)

// 2. Transformaciones
val palabrasRDD = rdd
  .flatMap(linea => linea.split(" ")) // Dividir frases en palabras
  .map(palabra => palabra.toLowerCase) // Convertir a minúsculas
  .filter(palabra => palabra.contains("s")) // Filtrar palabras que contienen 's'

// 3. Acción (Solo aquí se ejecuta el procesamiento)
val resultado = palabrasRDD.collect()

println("Palabras con 's':")
resultado.foreach(println)


Palabras con 's':
spark
es
spark
es
scala
spark
es


datos: Seq[String] = List(
  "Spark es rapido",
  "Spark es genial",
  "Scala y Spark",
  "Big Data es el futuro"
)
rdd: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[2] at parallelize at cmd5.sc:5
palabrasRDD: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[5] at filter at cmd5.sc:11
resultado: Array[String] = Array(
  "spark",
  "es",
  "spark",
  "es",
  "scala",
  "spark",
  "es"
)

## 3. Introducción al módulo Spark SQL (DataFrames)

Spark SQL permite consultar datos estructurados. El DataFrame es la abstracción principal aquí: es como una tabla en una base de datos relacional o un DataFrame en pandas, pero distribuido.

Los DataFrames utilizan el **Catalyst Optimizer** para optimizar automáticamente las consultas.

In [2]:
import spark.implicits._ // Import necesario para conversiones implícitas a DF

// Ejemplo Spark SQL: DataFrames

// 1. Crear DataFrame desde una secuencia de tuplas
val personas = Seq(
  ("Alice", 28, "Data Engineer"),
  ("Bob", 35, "Data Scientist"),
  ("Charlie", 23, "Data Analyst"),
  ("David", 42, "Data Engineer")
)

val df = personas.toDF("nombre", "edad", "rol")

// 2. Mostrar el esquema y los datos
df.printSchema()
df.show()

// 3. Consultas usando API de DataFrame
println("Data Engineers mayores de 25:")
df.filter($"rol" === "Data Engineer" && $"edad" > 25)
  .select("nombre", "edad")
  .show()

// 4. Agregaciones
println("Edad promedio por rol:")
df.groupBy("rol")
  .avg("edad")
  .show()

root
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = false)
 |-- rol: string (nullable = true)

+-------+----+--------------+
| nombre|edad|           rol|
+-------+----+--------------+
|  Alice|  28| Data Engineer|
|    Bob|  35|Data Scientist|
|Charlie|  23|  Data Analyst|
|  David|  42| Data Engineer|
+-------+----+--------------+

Data Engineers mayores de 25:
+------+----+
|nombre|edad|
+------+----+
| Alice|  28|
| David|  42|
+------+----+

Edad promedio por rol:
+--------------+---------+
|           rol|avg(edad)|
+--------------+---------+
| Data Engineer|     35.0|
|Data Scientist|     35.0|
|  Data Analyst|     23.0|
+--------------+---------+



import spark.implicits._
personas: Seq[(String, Int, String)] = List(
  ("Alice", 28, "Data Engineer"),
  ("Bob", 35, "Data Scientist"),
  ("Charlie", 23, "Data Analyst"),
  ("David", 42, "Data Engineer")
)
df: org.apache.spark.sql.package.DataFrame = [nombre: string, edad: int ... 1 more field]

In [7]:
// 5. Consultas SQL estándar
// Registramos el DataFrame como una vista temporal
df.createOrReplaceTempView("personas_view")

val sqlDF = spark.sql("SELECT rol, count(*) as total FROM personas_view GROUP BY rol")
sqlDF.show()

+--------------+-----+
|           rol|total|
+--------------+-----+
| Data Engineer|    2|
|Data Scientist|    1|
|  Data Analyst|    1|
+--------------+-----+



sqlDF: org.apache.spark.sql.package.DataFrame = [rol: string, total: bigint]

## 4. Ejercicios Prácticos (Sin resolver)

Usa estos ejercicios como base para practicar y documentar en tu porfolio.

### Ejercicio 1: Manipulación de DataFrames
Crea un DataFrame a partir de una lista de productos (nombre, precio, stock). Luego:
1. Añade una columna `valor_inventario` (precio * stock).
2. Filtra los productos que tengan un stock menor a 10.
3. Muestra el resultado.

In [4]:
// TODO: Definir datos y crear DataFrame
val productos = Seq(("Laptop", 1200.0, 5), ("Mouse", 25.0, 20), ("Teclado", 45.0, 15))

// TODO: Transformaciones
val dfProductos = productos.toDF("nombre", "precio", "stock")
val dfResultado = dfProductos
   .filter($"precio" > 30) // Filtrar productos con precio mayor a 30
   .withColumn("valor_total", $"precio" * $"stock") // Agregar columna de valor total
   .select("nombre", "valor_total") // Seleccionar solo nombre y valor total

dfResultado.show()

+-------+-----------+
| nombre|valor_total|
+-------+-----------+
| Laptop|     6000.0|
|Teclado|      675.0|
+-------+-----------+



productos: Seq[(String, Double, Int)] = List(
  ("Laptop", 1200.0, 5),
  ("Mouse", 25.0, 20),
  ("Teclado", 45.0, 15)
)
dfProductos: org.apache.spark.sql.package.DataFrame = [nombre: string, precio: double ... 1 more field]
dfResultado: org.apache.spark.sql.package.DataFrame = [nombre: string, valor_total: double]

### Ejercicio 2: Consultas SQL
Registra el DataFrame de productos anterior como una vista temporal y realiza una consulta SQL que devuelva el precio medio de los productos.

In [7]:
// TODO: Registrar vista y ejecutar SQL
dfProductos.createOrReplaceTempView("productos_view")
spark.sql("SELECT AVG(precio) AS promedio_precio FROM productos_view").show()

+-----------------+
|  promedio_precio|
+-----------------+
|423.3333333333333|
+-----------------+



In [8]:
spark.stop()